# Tarea #6 · Homografía y primer `posiciones.csv`

Convertimos la posición de los jugadores en el vídeo (píxeles) en su posición real sobre el campo (metros).

**Pasos del notebook**
1. Cargar los modelos de Roboflow (jugadores y puntos del campo).
2. Ver qué puntos del campo detecta el modelo en un fotograma y con qué confianza.
3. Calcular la homografía **con nuestro filtro** (confianza + RANSAC) y **como la demo** (sin filtro), y comparar.
4. Procesar el clip entero → `posiciones_todos.csv`, `homografia.csv` y vídeo de control.
5. Elegir un jugador → `posiciones.csv` (el formato que usa Alex, `docs/formatos.md`).
6. Experimento para la memoria: % de posiciones fuera del campo con y sin filtro.

**Antes de empezar**
- Kernel: el `.venv` del proyecto (en VS Code, arriba a la derecha → *Select Kernel* → `.venv`).
- El clip tiene que estar en `data/clips/` y los modelos en `models/`.
- Usamos siempre **rutas relativas** a la carpeta del repositorio, por las tildes de la ruta completa.

In [ ]:
import os, sys
from pathlib import Path

# El notebook está en notebooks/: trabajamos desde la raíz del repositorio
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")
print("Carpeta de trabajo:", Path.cwd().name)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import supervision as sv
import torch
from mplsoccer import Pitch
from ultralytics import YOLO

from src.fase2_homografia import (EstimadorHomografia, pixel_a_metros, pies_de_cajas,
                                  dentro_del_campo, LARGO_M, ANCHO_M)
from src.posiciones import (procesar_video, posiciones_de_un_jugador, posiciones_por_tramos, puntos_campo,
                            MODELO_JUGADORES, MODELO_CAMPO)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, "·", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "sin GPU")

In [ ]:
# ---- Configuración ----
VIDEO = "data/clips/2e57b9_0.mp4"      # ruta RELATIVA
SALIDA = Path("outputs") / Path(VIDEO).stem
FIG = Path("outputs/figuras"); FIG.mkdir(parents=True, exist_ok=True)
FRAME_EJEMPLO = 450                    # fotograma para las figuras (18 s)

assert Path(VIDEO).exists(), f"No encuentro {VIDEO}: copia el clip a data/clips/"
info = sv.VideoInfo.from_video_path(VIDEO)
print(f"{info.width}x{info.height}, {info.fps} fps, {info.total_frames} fotogramas ({info.total_frames/info.fps:.0f} s)")

## 1. Modelos y un fotograma de ejemplo

In [ ]:
modelo_jug = YOLO(MODELO_JUGADORES)
modelo_campo = YOLO(MODELO_CAMPO)
print("Clases del detector:", modelo_jug.names)

cap = cv2.VideoCapture(VIDEO)
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_EJEMPLO)
ok, frame = cap.read(); cap.release()
assert ok
rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)   # OpenCV lee en BGR
plt.figure(figsize=(14, 8)); plt.imshow(rgb); plt.axis("off"); plt.title(f"Fotograma {FRAME_EJEMPLO}"); plt.show()

## 2. Puntos del campo y su confianza

El modelo devuelve **siempre los 32 puntos**, también los que no se ven. Los que no se ven tienen confianza ≈ 0 y su posición es inventada. La demo de Roboflow los usaba igualmente.

In [ ]:
kp_xy, kp_conf = puntos_campo(modelo_campo, frame, DEVICE)
tabla = pd.DataFrame({"punto": range(1, 33), "u_px": kp_xy[:, 0].round(), "v_px": kp_xy[:, 1].round(),
                      "confianza": kp_conf.round(2)})
print(f"Puntos con confianza >= 0.5: {(kp_conf >= 0.5).sum()} de 32")

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(rgb); ax.axis("off")
buenos = kp_conf >= 0.5
ax.scatter(kp_xy[buenos, 0], kp_xy[buenos, 1], s=80, c="lime", edgecolors="k", label="confianza ≥ 0,5 (se usan)")
ax.scatter(kp_xy[~buenos, 0], kp_xy[~buenos, 1], s=80, c="red", marker="x", label="confianza < 0,5 (inventados)")
for i, (u, v) in enumerate(kp_xy):
    ax.annotate(str(i + 1), (u, v), xytext=(6, -6), textcoords="offset points", color="white", fontsize=9)
ax.legend(loc="upper left"); ax.set_title("Puntos del campo detectados")
plt.savefig(FIG / "puntos_campo_confianza.png", dpi=150, bbox_inches="tight"); plt.show()
tabla.T

## 3. Homografía: nuestro filtro frente a la demo

In [ ]:
nuestro = EstimadorHomografia(confianza_min=0.5, usar_ransac=True).actualizar(kp_xy, kp_conf)
demo = EstimadorHomografia(confianza_min=0.0, usar_ransac=False, error_max_cm=1e9).actualizar(kp_xy, kp_conf)
for nombre, r in [("Nuestro (filtro + RANSAC)", nuestro), ("Como la demo", demo)]:
    print(f"{nombre:28s} puntos usados {r.n_puntos:2d} · inliers {r.n_inliers:2d} · error medio {r.error_cm:7.1f} cm")

det = sv.Detections.from_ultralytics(modelo_jug(frame, imgsz=1280, device=DEVICE, verbose=False)[0])
det = det[det.class_id != 0]                      # sin el balón
pies = pies_de_cajas(det.xyxy)
xy_nuestro = pixel_a_metros(pies, nuestro.H)
xy_demo = pixel_a_metros(pies, demo.H)

pitch = Pitch(pitch_type="custom", pitch_length=LARGO_M, pitch_width=ANCHO_M, line_color="white", pitch_color="#3a7d44")
fig, axs = pitch.draw(nrows=1, ncols=2, figsize=(16, 6))
for ax, xy, titulo in [(axs[0], xy_nuestro, "Nuestro: confianza ≥ 0,5 + RANSAC"), (axs[1], xy_demo, "Como la demo: todos los puntos")]:
    fuera = ~dentro_del_campo(xy, margen_m=0)
    pitch.scatter(xy[~fuera, 0], xy[~fuera, 1], ax=ax, s=120, c="orange", edgecolors="k")
    ax.set_title(f"{titulo}\n{fuera.sum()} de {len(xy)} jugadores fuera del campo", fontsize=12)
plt.savefig(FIG / "homografia_filtro_vs_demo.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Procesar el clip entero

En la GPU tarda aproximadamente lo que dura el clip. Genera:
- `posiciones_todos.csv`: una fila por jugador y fotograma.
- `homografia.csv`: diagnóstico de la homografía en cada fotograma.
- `control.mp4`: vídeo con los puntos usados (verde), descartados por RANSAC (amarillo) o por confianza (rojo), y un radar.

In [ ]:
SALIDA.mkdir(parents=True, exist_ok=True)
todos, diag = procesar_video(VIDEO, device=DEVICE, video_control=SALIDA / "control.mp4")
todos.to_csv(SALIDA / "posiciones_todos.csv", index=False)
diag.to_csv(SALIDA / "homografia.csv", index=False)
print(f"H válida en {diag.valida.mean():.0%} de los fotogramas ({diag.reutilizada.sum()} reutilizando la anterior)")
print(f"Error medio de reproyección: {diag.error_cm.mean():.0f} cm · jugadores distintos: {todos.track_id.nunique()}")

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axs[0].plot(diag.t_s, diag.n_puntos, label="puntos con confianza ≥ 0,5"); axs[0].plot(diag.t_s, diag.n_inliers, label="inliers RANSAC")
axs[0].axhline(4, color="r", ls="--", lw=1, label="mínimo (4)"); axs[0].legend(); axs[0].set_ylabel("puntos")
axs[1].plot(diag.t_s, diag.error_cm); axs[1].set_ylabel("error (cm)")
axs[2].fill_between(diag.t_s, diag.valida, step="mid", alpha=.4, label="H válida")
axs[2].fill_between(diag.t_s, diag.reutilizada, step="mid", alpha=.6, label="reutilizada"); axs[2].legend(); axs[2].set_xlabel("tiempo (s)")
fig.suptitle("Calidad de la homografía a lo largo del clip")
plt.savefig(FIG / "homografia_calidad.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Elegir el jugador → `posiciones.csv`

De momento el jugador se elige a mano con su `track_id` (el clic llega en la tarea #9). Mira los números en `control.mp4`.

**Ojo con los cambios de identidad (ID switch):** cuando dos jugadores se cruzan y uno tapa al otro, el tracker puede intercambiarles el número. Si pasa, se unen los tramos a mano con `posiciones_por_tramos`.

In [ ]:
# Jugadores que más tiempo se ven (los números son los de control.mp4)
resumen = (todos[todos.valido == 1].groupby(["track_id", "clase"]).size()
           .rename("fotogramas_validos").reset_index().sort_values("fotogramas_validos", ascending=False))
resumen.head(15)

In [ ]:
# Delantero blanco en 2e57b9_0.mp4. El tracker le cambia el número al cruzarse con el
# defensa que lo marca (ID switch): es el 7 hasta el fotograma 281 y el 21 desde el 288.
# Entre 282 y 287 está tapado. La solución automática llega con la re-identificación (#9).
TRAMOS = [(7, None, 281), (21, 288, None)]
ATACA_IZQUIERDA = True   # los blancos atacan hacia la portería de la izquierda de la cámara
# Con ATACA_IZQUIERDA el campo se gira para que siempre ataque hacia x = 105 (docs/formatos.md)
TRACK_ID = TRAMOS[0][0]
pos = posiciones_por_tramos(todos, diag, TRAMOS, ataca_izquierda=ATACA_IZQUIERDA)
pos.to_csv(SALIDA / "posiciones.csv", index=False)
print(f"Jugador {TRACK_ID}: {pos.valido.mean():.0%} de fotogramas válidos → {SALIDA / 'posiciones.csv'}")

pitch = Pitch(pitch_type="custom", pitch_length=LARGO_M, pitch_width=ANCHO_M, line_color="white", pitch_color="#3a7d44")
fig, ax = pitch.draw(figsize=(10, 7))
v = pos[pos.valido == 1]
pitch.plot(v.x_m, v.y_m, ax=ax, color="orange", lw=2)
pitch.scatter(v.x_m.iloc[-1:], v.y_m.iloc[-1:], ax=ax, s=150, c="orange", edgecolors="k", zorder=3, label="fin")
pitch.scatter(v.x_m.iloc[:1], v.y_m.iloc[:1], ax=ax, s=150, c="white", edgecolors="k", zorder=3, label="inicio")
ax.set_title(f"Trayectoria del delantero (tracks {pos.track_id.iloc[0]}), atacando hacia la derecha"); ax.legend(loc="upper left")
plt.savefig(FIG / "trayectoria_delantero.png", dpi=150, bbox_inches="tight"); plt.show()
pos.head()

## 6. Experimento para la memoria: con filtro frente a como la demo

Procesamos el clip una segunda vez sin filtro ni RANSAC y comparamos cuántas posiciones caen fuera del campo. Tarda lo mismo que el paso 4.

In [ ]:
todos_demo, diag_demo = procesar_video(VIDEO, device=DEVICE, modo_demo=True, progreso=False)
def pct_fuera(df):
    return (df.valido == 0).mean()
comparacion = pd.DataFrame({
    "Nuestro (filtro + RANSAC)": [diag.n_puntos.mean(), diag.error_cm.median(), pct_fuera(todos)],
    "Como la demo":              [diag_demo.n_puntos.mean(), diag_demo.error_cm.median(), pct_fuera(todos_demo)],
}, index=["puntos usados por fotograma", "error de reproyección, mediana (cm)", "% posiciones no válidas o fuera del campo"])
comparacion.to_csv(FIG / "comparacion_filtro_vs_demo.csv")
comparacion.round(3)